# NGO Donation Data Quality Audit and PostgreSQL Pipeline

**Student:** Zainab Kausar  
**Student ID:** DC-238

This project uses PostgreSQL as the main database and Python for analysis.

## 1. Import libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sqlalchemy import create_engine


## 2. Connect to PostgreSQL

In [ ]:
user=os.getenv('POSTGRES_USER','postgres')
password=os.getenv('POSTGRES_PASSWORD','your_password')
host=os.getenv('POSTGRES_HOST','localhost')
port=os.getenv('POSTGRES_PORT','5432')
database=os.getenv('POSTGRES_DB','ngo_donations')
engine=create_engine(f'postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}')
print('PostgreSQL engine created.')


## 3. Inspect raw CSV

In [ ]:
raw=pd.read_csv('../data/ngo_donations_raw.csv')
print(raw.shape)
display(raw.head())


## 4. Missing-email audit in PostgreSQL

In [ ]:
q="SELECT COUNT(*) AS total_records, COUNT(*) FILTER (WHERE email IS NULL) AS missing_emails, ROUND(COUNT(*) FILTER (WHERE email IS NULL)*100.0/COUNT(*),2) AS missing_percentage FROM donations_raw;"
display(pd.read_sql(q,engine))


## 5. Duplicate audit in PostgreSQL

In [ ]:
q="SELECT donation_id, COUNT(*) AS record_count FROM donations_raw GROUP BY donation_id HAVING COUNT(*) > 1 ORDER BY record_count DESC;"
display(pd.read_sql(q,engine))


## 6. Read cleaned PostgreSQL table

In [ ]:
clean=pd.read_sql('SELECT * FROM donations_cleaned',engine)
print('Cleaned records:',len(clean))
display(clean.head())


## 7. Donations by payment method

In [ ]:
q="SELECT payment_method, COUNT(*) AS donation_count, SUM(amount) AS total_donations, ROUND(AVG(amount),2) AS average_donation FROM donations_cleaned GROUP BY payment_method ORDER BY total_donations DESC;"
payment_totals=pd.read_sql(q,engine)
display(payment_totals)


## 8. Monthly donations

In [ ]:
q="SELECT month, COUNT(*) AS donation_count, SUM(amount) AS total_donations FROM donations_cleaned GROUP BY month ORDER BY month;"
display(pd.read_sql(q,engine))


## 9. PostgreSQL data types

In [ ]:
q="SELECT table_name,column_name,data_type FROM information_schema.columns WHERE table_name IN ('donations_raw','donations_cleaned') ORDER BY table_name,ordinal_position;"
display(pd.read_sql(q,engine))


## 10. Outlier analysis in PostgreSQL

In [ ]:
q="WITH q AS (SELECT PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY amount) q1,PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY amount) q3 FROM donations_cleaned) SELECT d.* FROM donations_cleaned d,q WHERE d.amount > q.q3+1.5*(q.q3-q.q1) ORDER BY d.amount DESC;"
display(pd.read_sql(q,engine))


## 11. Visualizations

In [ ]:
plt.figure(figsize=(8,5)); payment_totals.set_index('payment_method')['total_donations'].sort_values().plot(kind='bar'); plt.title('Total Donations by Payment Method'); plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(8,5)); plt.hist(clean['amount'],bins=25); plt.title('Distribution of Donation Amounts'); plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(7,5)); plt.boxplot(clean['amount']); plt.title('Donation Amount Outliers'); plt.tight_layout(); plt.show()


In [ ]:
source=pd.read_csv('../data/ngo_donations_raw.csv'); plt.figure(figsize=(10,4)); plt.imshow(source.isna().astype(int).T,aspect='auto',interpolation='nearest'); plt.yticks(range(len(source.columns)),source.columns); plt.xticks([]); plt.title('Missing Values Before Cleaning'); plt.tight_layout(); plt.show()


## 12. Final takeaways

PostgreSQL stores the raw and cleaned data. SQL performs auditing, cleaning and aggregation. Python provides the visualization layer. Exact duplicates are removed, missing emails are retained as `Unknown`, amounts and dates are converted to appropriate types, and unusually large donations are flagged for review.